In [1]:
import sys
import os
sys.path.append("..")

In [2]:
import pickle
import pandas as pd
import numpy as np
import statsmodels.api as sm


In [3]:
from src.data_preprocessing import LoanDataPreprocessor
from src.feature_engineering import apply_feature_engineering

In [4]:
print(os.getcwd())

C:\Users\welcome\Credit risk project\pipeline


In [5]:
with open('../artifacts/pd_model.pkl','rb') as f:
    model=pickle.load(f)

with open('../artifacts/woe_binner.pkl', 'rb') as f:
    woe=pickle.load(f)

with open('../artifacts/final_features.pkl', 'rb') as f:
    final_features=pickle.load(f)

In [6]:
data_processor=LoanDataPreprocessor()

INFO:root:Preprocessing version: v1.0


In [7]:
df_dev=data_processor.preprocess('../data/loan_data_2007_2014.csv')

INFO:root:Cleaning employment length
INFO:root:Cleaning term
INFO:root:dates conversion
INFO:root:credit age creation
INFO:root:missing values handling
INFO:root:target variable creation
INFO:root:Applying outlier capping on 33 columns
INFO:root:final dataset shape: (466285, 81)


In [8]:
df_dev_fe,feature_cols=apply_feature_engineering(df_dev, None)

INFO:root:Starting feature engineering
INFO:root:Applying categorical grouping
INFO:root:Feature engineering completed. Shape: (466285, 86)
INFO:root:Feature engineering version: v1.0


In [9]:
df_dev_woe=woe.apply_bins(df_dev_fe)

In [10]:
df_dev_woe=woe.transform(df_dev_woe)

In [11]:
final_features

['grade_woe',
 'inq_last_6mths_bin_woe',
 'annual_inc_bin_woe',
 'purpose_woe',
 'credit_age_months_bin_woe',
 'dti_bin_woe',
 'initial_list_status_woe',
 'home_ownership_woe',
 'verification_status_woe']

In [12]:
df_dev_woe=df_dev_woe[final_features]

In [13]:
X_dev=sm.add_constant(df_dev_woe)
X_dev=X_dev[['const']+final_features]
df_dev_woe['pd']=model.predict_proba(X_dev)

INFO:root:Generating predictions
INFO:root:scoring data shape : (466285, 10)


In [14]:
df_dev=df_dev_woe

In [15]:
df_oot=data_processor.preprocess('../data/loan_data_2015.csv')

INFO:root:Cleaning employment length
INFO:root:Cleaning term
INFO:root:dates conversion
INFO:root:credit age creation
INFO:root:missing values handling
INFO:root:target variable creation
INFO:root:Applying outlier capping on 47 columns
INFO:root:final dataset shape: (421094, 80)


In [16]:
df_oot.shape

(421094, 80)

In [17]:
pd.options.display.max_rows=None

In [18]:
df_oot.isnull().sum()

id                             0
member_id                      0
loan_amnt                      0
funded_amnt                    0
funded_amnt_inv                0
term                           0
int_rate                       0
installment                    0
grade                          0
sub_grade                      0
emp_title                      0
emp_length                     0
home_ownership                 0
annual_inc                     0
verification_status            0
issue_d                        0
loan_status                    0
pymnt_plan                     0
url                            0
desc                           0
purpose                        0
title                          0
zip_code                       0
addr_state                     0
dti                            0
delinq_2yrs                    0
earliest_cr_line               0
inq_last_6mths                 0
mths_since_last_delinq         0
mths_since_last_record         0
open_acc  

In [19]:
#df=woe_binner.apply_bins(df_oot)

In [20]:
df_oot_fe, feature_cols=apply_feature_engineering(df_oot, None)

INFO:root:Starting feature engineering
INFO:root:Applying categorical grouping
INFO:root:Feature engineering completed. Shape: (421094, 85)
INFO:root:Feature engineering version: v1.0


In [21]:
df_oot_fe.shape

(421094, 85)

In [22]:
print(df_oot_fe.columns)

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title',
       'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
       'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose',
       'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs',
       'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq',
       'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'initial_list_status', 'out_prncp',
       'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
       'total_rec_int', 'total_rec_late_fee', 'recoveries',
       'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt',
       'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med',
       'mths_since_last_major_derog', 'policy_code', 'application_type',
       'annual_inc_joint', 'dti_joint', 'verification_status_joint',
    

In [23]:
df_oot_fe.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,term_int,earliest_cr_line_date,credit_age_months,loan_status_clean,good_bad,home_ownership_grp,purpose_grp,addr_state_grp,mths_since_last_delinq_missing,mths_since_last_record_missing
0,68444620,73334399,35000,35000,35000.0,60 months,11.99,778.38,C,C1,...,60,1990-02-01,314,Issued,0,MORTGAGE,low_risk,LOW_RISK,0,0
1,68547583,73437441,8650,8650,8650.0,36 months,5.32,260.50,A,A1,...,36,2001-07-01,175,Issued,0,MORTGAGE,low_risk,MEDIUM_RISK,0,0
2,67849662,72708407,4225,4225,4225.0,36 months,14.85,146.16,C,C5,...,36,2011-07-01,53,Issued,0,RENT,medium_risk,LOW_RISK,0,0
3,68506885,73396712,10000,10000,10000.0,60 months,11.99,222.40,C,C1,...,60,1998-12-01,206,Issued,0,RENT,medium_risk,MEDIUM_RISK,0,0
4,68341763,72928789,20000,20000,20000.0,60 months,10.78,432.66,B,B4,...,60,2000-08-01,186,Issued,0,MORTGAGE,low_risk,LOW_RISK,0,0


In [24]:
df_oot_woe=woe.apply_bins(df_oot_fe)
df_oot_woe=woe.transform(df_oot_woe)
df_oot_woe.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,initial_list_status_woe,int_rate_bin_woe,dti_bin_woe,annual_inc_bin_woe,installment_bin_woe,delinq_2yrs_bin_woe,inq_last_6mths_bin_woe,credit_age_months_bin_woe,open_acc_bin_woe,total_acc_bin_woe
0,68444620,73334399,35000,35000,35000.0,60 months,11.99,778.38,C,C1,...,0.223494,0.385109,0.235439,0.428106,-0.068403,-0.000472,0.195875,0.216844,0.043296,0.090641
1,68547583,73437441,8650,8650,8650.0,36 months,5.32,260.50,A,A1,...,0.223494,1.052007,0.199861,0.278822,-0.002328,-0.000472,0.195875,0.018155,-0.003823,0.033141
2,67849662,72708407,4225,4225,4225.0,36 months,14.85,146.16,C,C5,...,0.223494,-0.060795,0.083862,-0.197276,0.086846,0.004014,0.195875,-0.213277,-0.005804,-0.188009
3,68506885,73396712,10000,10000,10000.0,60 months,11.99,222.40,C,C1,...,0.223494,0.385109,-0.203865,-0.197276,0.086846,-0.000472,-0.062870,0.131776,-0.005804,0.033141
4,68341763,72928789,20000,20000,20000.0,60 months,10.78,432.66,B,B4,...,0.223494,0.385109,0.199861,0.022372,-0.002328,-0.000472,0.195875,0.018155,-0.005804,-0.059072


In [25]:
final_features

['grade_woe',
 'inq_last_6mths_bin_woe',
 'annual_inc_bin_woe',
 'purpose_woe',
 'credit_age_months_bin_woe',
 'dti_bin_woe',
 'initial_list_status_woe',
 'home_ownership_woe',
 'verification_status_woe']

In [26]:
#df_oot_woe=df_oot_woe[final_features]

In [27]:
X_oot=df_oot_woe[final_features].copy()

In [28]:
X_oot=sm.add_constant(X_oot)

In [29]:
X_oot=X_oot[['const']+final_features]

In [30]:
df_oot_woe['pd']=model.predict_proba(X_oot)

INFO:root:Generating predictions
INFO:root:scoring data shape : (421094, 10)


In [31]:
#df_dev=df_dev[final_features+ ['pd']]

In [32]:
df_oot_woe=df_oot_woe[final_features+ ['pd']]

In [33]:
df_oot_woe.head()

,grade_woe,inq_last_6mths_bin_woe,annual_inc_bin_woe,purpose_woe,credit_age_months_bin_woe,dti_bin_woe,initial_list_status_woe,home_ownership_woe,verification_status_woe,pd
0,-0.056491,0.195875,0.428106,0.126488,0.216844,0.235439,0.223494,0.146051,0.053403,0.043398
1,1.118678,0.195875,0.278822,0.274140,0.018155,0.199861,0.223494,0.146051,0.170388,0.018037
2,-0.056491,0.195875,-0.197276,-0.043086,-0.213277,0.083862,0.223494,-0.165189,0.053403,0.110000
3,-0.056491,-0.062870,-0.197276,-0.043086,0.131776,-0.203865,0.223494,-0.165189,0.170388,0.117374
4,0.373511,0.195875,0.022372,0.126488,0.018155,0.199861,0.223494,0.146051,0.170388,0.043127


In [34]:
def calculate_psi(expected, actual, bins=10):

    # CASE 1: DISCRETE / LOW UNIQUE
    if expected.dtype == 'object' or expected.nunique() < bins:

        all_vals = sorted(set(expected.unique()).union(set(actual.unique())))

        expected_dist = expected.value_counts(normalize=True).reindex(all_vals, fill_value=0)
        actual_dist   = actual.value_counts(normalize=True).reindex(all_vals, fill_value=0)

        psi_df = pd.DataFrame({
            'expected': expected_dist,
            'actual': actual_dist
        })

    else:
        # CASE 2: CONTINUOUS
        expected_bins = pd.qcut(expected, q=bins, duplicates='drop')
        actual_bins = pd.cut(actual, bins=expected_bins.cat.categories)

        expected_dist = expected_bins.value_counts(normalize=True).sort_index()
        actual_dist   = actual_bins.value_counts(normalize=True).sort_index()

        psi_df = pd.DataFrame({
            'expected': expected_dist,
            'actual': actual_dist
        }).fillna(0)

    # Avoid zeros
    psi_df['expected'] = psi_df['expected'].replace(0, 1e-6)
    psi_df['actual']   = psi_df['actual'].replace(0, 1e-6)

    # PSI formula
    psi_df['psi'] = (psi_df['expected'] - psi_df['actual']) * \
                    np.log(psi_df['expected'] / psi_df['actual'])

    return psi_df, psi_df['psi'].sum()

In [35]:
#score PSI
psi_table, score_psi=calculate_psi(df_dev['pd'],df_oot_woe['pd'])

In [36]:
psi_table

,expected,actual,psi
pd,,,
"(0.012899999999999998, 0.0391]",0.100001,0.156723,0.025485
"(0.0391, 0.0557]",0.100001,0.113070,0.001605
"(0.0557, 0.0704]",0.100010,0.111571,0.001265
"(0.0704, 0.0841]",0.099999,0.105672,0.000313
"(0.0841, 0.098]",0.099992,0.092186,0.000635
"(0.098, 0.113]",0.099999,0.085855,0.002157
"(0.113, 0.132]",0.100014,0.089412,0.001188
"(0.132, 0.156]",0.099997,0.083618,0.002930
"(0.156, 0.194]",0.099986,0.084314,0.002672


In [37]:
score_psi

np.float64(0.043941813576220366)

In [38]:
feature_cols = final_features

for col in feature_cols:

    missing_in_oot = set(df_dev[col].unique()) - set(df_oot_woe[col].unique())
    missing_in_dev = set(df_oot_woe[col].unique()) - set(df_dev[col].unique())

    if missing_in_oot or missing_in_dev:

        print(f"{col} shows population distribution differences between DEV and OOT datasets.")

        if missing_in_oot:
            print("Categories present in DEV but absent in OOT:", missing_in_oot)

        if missing_in_dev:
            print("Categories present in OOT but absent in DEV:", missing_in_dev)

        print("-" * 80)

credit_age_months_bin_woe shows population distribution differences between DEV and OOT datasets.
Categories present in DEV but absent in OOT: {np.float64(0.0)}
--------------------------------------------------------------------------------
home_ownership_woe shows population distribution differences between DEV and OOT datasets.
Categories present in DEV but absent in OOT: {np.float64(-0.8561419809643181), np.float64(-0.3723450295862467)}
--------------------------------------------------------------------------------


In [39]:

feature_psi={}
feature_cols=final_features

for col in feature_cols:
    expected=df_dev[col]
    actual=df_oot_woe[col]

         
    _, psi_val=calculate_psi(expected, actual)
    feature_psi[col]=psi_val

feature_psi_df=pd.DataFrame.from_dict(feature_psi, orient='index', columns=['PSI']).sort_values(by='PSI', ascending=False)
feature_psi_df

,PSI
purpose_woe,3.130525
initial_list_status_woe,0.333003
inq_last_6mths_bin_woe,0.053415
verification_status_woe,0.048312
dti_bin_woe,0.043434
credit_age_months_bin_woe,0.013183
home_ownership_woe,0.006986
grade_woe,0.006735
annual_inc_bin_woe,0.005028


In [40]:
def psi_alert_label(psi):
    if psi <0.1:
        return 'Stable'
    elif psi <0.25:
        return 'Moderate shift'
    else:
        return 'Significant Drift'

In [41]:
feature_psi_df['alert']=feature_psi_df['PSI'].apply(psi_alert_label)

In [42]:
def monitoring_action(psi):
    if psi<0.1:
        return 'No action'
    elif psi <0.25:
        return 'Monitor'
    else:
        return 'Investigate'


In [43]:
feature_psi_df['action']=feature_psi_df['PSI'].apply(monitoring_action)

In [44]:
feature_psi_df

,PSI,alert,action
purpose_woe,3.130525,Significant Drift,Investigate
initial_list_status_woe,0.333003,Significant Drift,Investigate
inq_last_6mths_bin_woe,0.053415,Stable,No action
verification_status_woe,0.048312,Stable,No action
dti_bin_woe,0.043434,Stable,No action
credit_age_months_bin_woe,0.013183,Stable,No action
home_ownership_woe,0.006986,Stable,No action
grade_woe,0.006735,Stable,No action
annual_inc_bin_woe,0.005028,Stable,No action


In [45]:
top_drift=feature_psi_df[feature_psi_df['PSI']>0.25]
print(top_drift)

                              PSI              alert       action
purpose_woe              3.130525  Significant Drift  Investigate
initial_list_status_woe  0.333003  Significant Drift  Investigate


In [46]:
summary=feature_psi_df['alert'].value_counts()
print(summary)

alert
Stable               7
Significant Drift    2
Name: count, dtype: int64


In [47]:
feature_psi_df.to_csv('../reports/feature_psi.csv')

In [48]:
score_alert=psi_alert_label(score_psi)
score_action=monitoring_action(score_psi)
print(f'score alert:{score_alert} and action:{score_action}')

score alert:Stable and action:No action


In [49]:
model_version="V1.0"
monitoring_date=pd.Timestamp.today()
monitoring_date

Timestamp('2026-05-13 15:45:16.160333')

In [50]:
score_psi_df=pd.DataFrame({'metric':['score_psi'], 'value':[score_psi]})
score_psi_df.to_csv('../reports/score_psi.csv', index=False)

In [51]:
monitoring_summary = {
    "score_psi": score_psi,
    "score_alert": score_alert,
    "score_action": score_action,
    "num_features_high_drift": len(top_drift),
    "monitoring_date": monitoring_date
}

print(monitoring_summary)

{'score_psi': np.float64(0.043941813576220366), 'score_alert': 'Stable', 'score_action': 'No action', 'num_features_high_drift': 2, 'monitoring_date': Timestamp('2026-05-13 15:45:16.160333')}


### Governance Framework

#### Model Lifecycle

Development → Validation → Approval → Deployment → Monitoring → Periodic Review

#### Roles & Responsibilities

* Model Developer: Model development
* Model Validator: Independent validation
* Risk Committee: Model approval
* Business Team: Model usage

#### Model Inventory

* Model Name: PD Model
* Version: V1.0
* Owner: Risk Analytics Team
* Monitoring Date: Current run date


##### Independent validation is performed prior to model approval and deployment.
---

### Model Monitoring Analysis

**1. Score Stability**  
The Population Stability Index(PSI) for predicted probabilites (PD) is 0.043.

**Interpretation:**
- PSI < 0.1 indicates stable model performance 
- No significant shift observed in score distribution

#### Conclusion: 
The model remains stable on the out-of-time monitoring population with no immediate recalibration required.
  
### 2. Feature-Level Stability

Feature-level PSI analysis indicates that most model variables remained stable between the development and out-of-time (OOT) datasets. However, two variables exhibited significant population drift:

• **Initial List Status (PSI = 0.33)**  
  Significant drift was observed in the distribution of initial loan listing status between the development and OOT populations.

  This may reflect changes in loan origination practices, platform listing policies, or borrower acquisition strategy over time.

  Despite the observed shift, the variable continued to maintain economically consistent directional behavior within the final model.

• **Purpose (PSI = 3.13)**  
  Significant population drift was observed in loan purpose distribution between the development and OOT datasets.

  This may reflect changes in borrower loan usage patterns, platform lending strategy, product mix, or broader economic conditions over time.

  Although substantial drift was identified in this variable, overall score PSI remained stable, indicating that portfolio-level prediction behavior was not materially impacted.

• **Remaining Features**  
  Remaining model features exhibited PSI values below 0.1, indicating stable distributions with no material population drift detected.  

  
 **3. Key Insight**  
 Although certain features exhibit drift, the overall model remains stable as indicated by low PD PSI.

 This suggests:  
  - Model is robust to underlying data shifts  
  - Drift has not materially impacted prediction behavior

  **4. Action plan**
 Trigger enhanced review or recalibration assessment if:
- Score PSI exceeds 0.1 persistently
- OR multiple key features exceed PSI > 0.25
- OR material performance degradation is observed
    
**5. Monitoring Framework**

A monitoring framework is established to ensure ongoing performance and stability of the model post-deployment.

#### Key Monitoring Metrics:

* **Population Stability Index (PSI):**
  Used to detect shifts in input features and score distribution over time.

* **Model Performance Metrics:**
  Periodic evaluation of AUC and default rates as actual outcomes become available.

* **Calibration Monitoring:**
  Comparison of predicted PD vs observed default rates to ensure calibration remains valid.


* **Scenario-Based Monitoring Review:**
  Periodic assessment of model sensitivity under adverse borrower conditions.

#### Monitoring Frequency:

* Monthly monitoring of PSI and score distribution
* Quarterly review of model performance and calibration

#### Thresholds:

* PSI < 0.1 → Stable
* PSI 0.1–0.25 → Moderate shift
* PSI > 0.25 → Significant drift (action required)

#### Actions:

* Investigate data drift
* Recalibrate model if necessary
* Redevelop model if performance degrades significantly


**Limitation:**
- PSI captures distribution shift but not performance degradation directly
- Requires complementary metrics like AUC/KS monitoring
-  Monitoring results are based on historical OOT data and may not fully represent future macroeconomic regime changes.

All monitoring metrics are computed using a consistent data transformation pipeline, ensuring reliable comparison across development and production datasets.

#### Monitoring Conclusion

The monitoring analysis indicates that the model remains stable on the out-of-time population, with low overall score PSI and acceptable feature-level stability for most variables.

Although isolated feature drift was identified in specific variables, the overall score distribution remains stable, suggesting that model prediction behavior has not materially deteriorated.

No immediate recalibration or redevelopment action is required based on current monitoring results. Continued periodic monitoring is recommended to ensure ongoing model stability and performance.



  

  